# Nível 1 — Dados e primeira análise com LLM

Triagem AML: tratamento de dados, regras determinísticas (pandas) e parecer de um LLM sobre
um cliente sinalizado. Base: `dados/dados_nivel_1.json` (20 operações, 6 clientes).

In [1]:
import json
import os
import time
import pandas as pd
from pydantic import BaseModel, ValidationError
from typing import Literal

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

with open('../dados/dados_nivel_1.json', encoding='utf-8') as f:
    raw = json.load(f)

taxa_cambio_usd_brl = raw['taxa_cambio_usd_brl']
df = pd.DataFrame(raw['operacoes'])
print('taxa_cambio_usd_brl:', taxa_cambio_usd_brl)
df

taxa_cambio_usd_brl: 5.4


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


## Parte A.2 — Diagnóstico de qualidade dos dados

Antes de tratar, inspecionamos o que existe: duplicatas, valores nulos e moedas presentes.

In [2]:
print('Linhas:', len(df))
print('IDs duplicados (linha inteira repetida):', df.duplicated(subset=['id']).sum())
display_dup = df[df.duplicated(subset=['id'], keep=False)]
display_dup

Linhas: 20
IDs duplicados (linha inteira repetida): 1


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [3]:
print('Nulos por coluna:')
print(df.isna().sum())
df[df['data'].isna()]

Nulos por coluna:
id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,NaN,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


In [4]:
print('Moedas presentes:')
print(df['moeda'].value_counts())
df[df['moeda'] != 'BRL']

Moedas presentes:
moeda
BRL    19
USD     1
Name: count, dtype: int64


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
13,OP-0013,CLI-A-4,2026-03-24,12000,USD,ted,transferencia_recebida,Zeta Importacao,remessa internacional


## O que encontramos e por que tratamos assim

1. **Operação `OP-0007` duplicada** (linha idêntica repetida duas vezes para o mesmo cliente,
   mesma data, mesmo valor, mesma contraparte). Tudo indica erro de duplicação na extração do
   sistema legado, não duas operações distintas com o mesmo ID por coincidência — um ID de
   operação deveria ser único. **Decisão:** remover a duplicata, mantendo uma única ocorrência
   (`drop_duplicates(subset='id')`). Isso é essencial: se não removêssemos, o cliente `CLI-A-3`
   ficaria com soma diária inflada artificialmente e poderia disparar a Regra 1 (fracionamento)
   por um erro de dado, não por comportamento real — mostramos isso explicitamente na seção de
   validação mais abaixo.

2. **Uma operação (`OP-0017`) sem `data`** (valor `null`), com a observação explícita
   `"data nao capturada pelo sistema"` — ou seja, o próprio dado já indica que é uma falha de
   captura, não uma operação sem data por natureza. **Decisão:** não descartamos a linha (o
   valor e o cliente são informações válidas e reais), mas a **excluímos do cálculo da Regra 1**
   (fracionamento), que depende de agrupar por data — não é seguro presumir uma data. Ela
   continua entrando nas agregações de volume total e de contagem por canal, e na Regra 2 (valor
   atípico), que não depende de data. Alternativa descartada: imputar uma data (ex.: média do
   cliente) — rejeitada porque criaria uma data artificial em um contexto de AML, o que é
   arriscado (poderia mascarar ou fabricar padrões).

3. **Uma operação em `USD`** (`OP-0013`, cliente `CLI-A-4`). O enunciado pede para normalizar
   todos os valores para BRL usando a taxa fixa fornecida no próprio arquivo
   (`taxa_cambio_usd_brl`). **Decisão:** criamos uma coluna `valor_brl` = `valor` para operações
   em BRL, e `valor * taxa_cambio_usd_brl` para operações em USD. Não convertemos comparando com
   cotação externa, conforme instruído.

Não removemos outliers de valor nem tentamos "corrigir" contrapartes/canais — isso é
exatamente o que a Regra 2 e o parecer do LLM devem avaliar, não algo para normalizar antes.

In [5]:
df_clean = df.drop_duplicates(subset=['id']).copy()
df_clean['data'] = pd.to_datetime(df_clean['data'], errors='coerce')
df_clean['data_valida'] = df_clean['data'].notna()

df_clean['valor_brl'] = df_clean.apply(
    lambda r: r['valor'] * taxa_cambio_usd_brl if r['moeda'] == 'USD' else r['valor'],
    axis=1,
)

print(f'Linhas antes: {len(df)} -> depois de remover duplicata: {len(df_clean)}')
df_clean[['id', 'cliente_id', 'data', 'data_valida', 'valor', 'moeda', 'valor_brl']]

Linhas antes: 20 -> depois de remover duplicata: 19


,id,cliente_id,data,data_valida,valor,moeda,valor_brl
0,OP-0001,CLI-A-1,2026-03-09,True,18100,BRL,18100.0
1,OP-0002,CLI-A-1,2026-03-09,True,17300,BRL,17300.0
2,OP-0003,CLI-A-1,2026-03-09,True,18800,BRL,18800.0
3,OP-0004,CLI-A-1,2026-03-21,True,3300,BRL,3300.0
4,OP-0005,CLI-A-2,2026-03-14,True,25900,BRL,25900.0
5,OP-0006,CLI-A-2,2026-03-14,True,27000,BRL,27000.0
6,OP-0007,CLI-A-3,2026-03-05,True,17200,BRL,17200.0
7,OP-0008,CLI-A-3,2026-03-05,True,15200,BRL,15200.0
8,OP-0009,CLI-A-3,2026-03-05,True,16100,BRL,16100.0
10,OP-0010,CLI-A-4,2026-03-03,True,3800,BRL,3800.0


## Parte A.4 — Agregações

In [6]:
volume_por_cliente = df_clean.groupby('cliente_id')['valor_brl'].sum().sort_values(ascending=False)
volume_por_cliente.rename('volume_total_brl').to_frame()

,volume_total_brl
cliente_id,
CLI-A-4,79500.0
CLI-A-1,57500.0
CLI-A-2,52900.0
CLI-A-3,48500.0
CLI-A-5,16900.0
CLI-A-6,10200.0


In [7]:
ops_por_canal = df_clean['canal'].value_counts()
ops_por_canal.rename('qtd_operacoes').to_frame()

,qtd_operacoes
canal,
pix,8
ted,5
boleto,3
cartao,2
especie,1


## Parte A.5 — Regras determinísticas

**Regra 1 — Fracionamento.** Cliente que, em uma mesma data, tem 3+ operações cuja soma em BRL
ultrapassa R$ 50.000,00, **e** nenhuma dessas operações isoladas atinge R$ 20.000,00. Usamos
apenas linhas com `data_valida == True` (ver justificativa acima). "Ultrapassa" e "atinge"
foram tratados como estritamente `>` e `>=`, respectivamente — limite é o valor que já conta
como atingido.

**Regra 2 — Valor atípico.** Operação cujo `valor_brl` é superior a 5× a mediana dos
`valor_brl` daquele cliente, aplicada apenas a clientes com 4+ operações (todas as operações do
cliente, independente de `data_valida`, pois a regra não depende de data).

In [8]:
def flag_fracionamento(df_clean):
    elegivel = df_clean[df_clean['data_valida']]
    grp = elegivel.groupby(['cliente_id', 'data'])['valor_brl']
    soma = grp.sum()
    qtd = grp.count()
    max_individual = grp.max()

    candidatos = pd.DataFrame({
        'soma': soma, 'qtd': qtd, 'max_individual': max_individual,
    }).reset_index()

    flagged = candidatos[
        (candidatos['qtd'] >= 3)
        & (candidatos['soma'] > 50000)
        & (candidatos['max_individual'] < 20000)
    ]
    return flagged

fracionamento = flag_fracionamento(df_clean)
fracionamento

,cliente_id,data,soma,qtd,max_individual
0,CLI-A-1,2026-03-09,54200.0,3,18800.0


In [9]:
clientes_fracionamento = set(fracionamento['cliente_id'])
df_clean['flag_fracionamento'] = df_clean['cliente_id'].isin(clientes_fracionamento)
print('Clientes sinalizados (Regra 1 - fracionamento):', clientes_fracionamento)

Clientes sinalizados (Regra 1 - fracionamento): {'CLI-A-1'}


In [10]:
def flag_valor_atipico(df_clean):
    contagem = df_clean.groupby('cliente_id')['id'].transform('count')
    elegivel = df_clean[contagem >= 4].copy()
    mediana = elegivel.groupby('cliente_id')['valor_brl'].transform('median')
    elegivel['limite_atipico'] = mediana * 5
    elegivel['atipico'] = elegivel['valor_brl'] > elegivel['limite_atipico']
    return elegivel[['id', 'cliente_id', 'valor_brl', 'limite_atipico', 'atipico']]

atipicos = flag_valor_atipico(df_clean)
atipicos[atipicos['atipico']]

,id,cliente_id,valor_brl,limite_atipico,atipico
13,OP-0013,CLI-A-4,64800.0,27250.0,True


In [11]:
df_clean = df_clean.merge(
    atipicos[['id', 'atipico']].rename(columns={'atipico': 'flag_valor_atipico'}),
    on='id', how='left',
)
df_clean['flag_valor_atipico'] = df_clean['flag_valor_atipico'].fillna(False)
df_clean[['id', 'cliente_id', 'valor_brl', 'moeda', 'flag_fracionamento', 'flag_valor_atipico']]

,id,cliente_id,valor_brl,moeda,flag_fracionamento,flag_valor_atipico
0,OP-0001,CLI-A-1,18100.0,BRL,True,False
1,OP-0002,CLI-A-1,17300.0,BRL,True,False
2,OP-0003,CLI-A-1,18800.0,BRL,True,False
3,OP-0004,CLI-A-1,3300.0,BRL,True,False
4,OP-0005,CLI-A-2,25900.0,BRL,False,False
5,OP-0006,CLI-A-2,27000.0,BRL,False,False
6,OP-0007,CLI-A-3,17200.0,BRL,False,False
7,OP-0008,CLI-A-3,15200.0,BRL,False,False
8,OP-0009,CLI-A-3,16100.0,BRL,False,False
9,OP-0010,CLI-A-4,3800.0,BRL,False,False


## Parte A.6 — Validação da Regra 1

Comparamos dois clientes com o mesmo padrão superficial (3 operações no mesmo dia, mesmo
formato) para mostrar que a regra captura o caso certo e não o caso parecido:

- **`CLI-A-1`** — 3 operações em `2026-03-09`, soma R$ 54.200,00, nenhuma isolada atinge
  R$ 20.000,00 → **deve ser sinalizado**.
- **`CLI-A-3`** — após remover a duplicata de `OP-0007`, restam 3 operações em `2026-03-05`,
  soma R$ 48.500,00 (abaixo do limite de R$ 50.000,00) → **não deve ser sinalizado**. Se a
  duplicata não tivesse sido removida, a soma teria ficado em R$ 65.700,00 e o cliente seria
  falsamente sinalizado — evidência direta de por que a deduplicação da Parte A.2 importa.

In [12]:
caso_positivo = df_clean[df_clean['cliente_id'] == 'CLI-A-1'][
    ['id', 'data', 'valor_brl', 'flag_fracionamento']
]
caso_negativo = df_clean[df_clean['cliente_id'] == 'CLI-A-3'][
    ['id', 'data', 'valor_brl', 'flag_fracionamento']
]
print('CLI-A-1 (deve sinalizar):')
display(caso_positivo)
print(f"  soma = {caso_positivo['valor_brl'].sum():.2f}")

print('\nCLI-A-3 (não deve sinalizar, mesmo padrão superficial):')
display(caso_negativo)
print(f"  soma = {caso_negativo['valor_brl'].sum():.2f}")

assert df_clean.loc[df_clean['cliente_id'] == 'CLI-A-1', 'flag_fracionamento'].any()
assert not df_clean.loc[df_clean['cliente_id'] == 'CLI-A-3', 'flag_fracionamento'].any()
print('\nOK: Regra 1 captura CLI-A-1 e não captura CLI-A-3.')

CLI-A-1 (deve sinalizar):


,id,data,valor_brl,flag_fracionamento
0,OP-0001,2026-03-09,18100.0,True
1,OP-0002,2026-03-09,17300.0,True
2,OP-0003,2026-03-09,18800.0,True
3,OP-0004,2026-03-21,3300.0,True


  soma = 57500.00

CLI-A-3 (não deve sinalizar, mesmo padrão superficial):


,id,data,valor_brl,flag_fracionamento
6,OP-0007,2026-03-05,17200.0,False
7,OP-0008,2026-03-05,15200.0,False
8,OP-0009,2026-03-05,16100.0,False


  soma = 48500.00

OK: Regra 1 captura CLI-A-1 e não captura CLI-A-3.


## Parte A.6 (cont.) — Validação da Regra 2

Mesma lógica de validação: mostrar que a regra captura o caso certo e **não** captura casos
parecidos que não se enquadram.

- **`CLI-A-4` — deve sinalizar.** `OP-0013` (USD 12.000 → R$ 64.800) supera 5× a mediana do
  cliente (5 × R$ 5.450 = R$ 27.250).
- **`CLI-A-2` — não deve sinalizar**, apesar de ter operações de R$ 25.900 e R$ 27.000, valores
  *absolutos* maiores que a mediana de vários outros clientes. A Regra 2 mede desvio **relativo
  ao próprio histórico**: para esse cliente os dois valores são homogêneos, então nenhum é
  atípico. É o caso que separa "valor grande" de "valor fora do padrão do cliente".
- **`CLI-A-3` — fora do escopo da regra**: sobram 3 operações após a deduplicação, abaixo do
  mínimo de 4 exigido pelo enunciado. Note que **antes** de remover a duplicata de `OP-0007` ele
  teria 4 linhas e entraria no cálculo — a limpeza da Parte A.2 muda a elegibilidade, não só os
  totais.

O ponto mais importante desta célula é a última verificação: se tivéssemos aplicado a regra
sobre `valor` em vez de `valor_brl`, `OP-0013` valeria 12.000 e ficaria **abaixo** do limite de
27.250 — a única operação atípica da base passaria despercebida. A normalização de moeda da
Parte A.3 não é formalidade: é o que torna a Regra 2 capaz de enxergar esse caso.

In [13]:
resumo_regra2 = df_clean.groupby('cliente_id').agg(
    qtd_operacoes=('id', 'count'),
    mediana_brl=('valor_brl', 'median'),
    maior_valor_brl=('valor_brl', 'max'),
)
resumo_regra2['limite_5x'] = resumo_regra2['mediana_brl'] * 5
resumo_regra2['elegivel_4_ops'] = resumo_regra2['qtd_operacoes'] >= 4
resumo_regra2['sinalizado'] = df_clean.groupby('cliente_id')['flag_valor_atipico'].any()
display(resumo_regra2)

# 1) caso positivo: CLI-A-4
assert df_clean.loc[df_clean['cliente_id'] == 'CLI-A-4', 'flag_valor_atipico'].any()

# 2) caso parecido que NAO se enquadra: CLI-A-2 tem valores altos em termos absolutos,
#    mas homogeneos para o proprio historico
assert not df_clean.loc[df_clean['cliente_id'] == 'CLI-A-2', 'flag_valor_atipico'].any()

# 3) guarda de 4+ operacoes: CLI-A-3 fica de fora (3 operacoes apos a deduplicacao)
assert not df_clean.loc[df_clean['cliente_id'] == 'CLI-A-3', 'flag_valor_atipico'].any()
assert (df_clean['cliente_id'] == 'CLI-A-3').sum() == 3

print('OK: Regra 2 sinaliza CLI-A-4 e nao sinaliza CLI-A-2 (valor alto porem tipico)')
print('OK: CLI-A-3 fica fora da regra por ter menos de 4 operacoes apos a limpeza')

# 4) a conversao de moeda e o que torna OP-0013 detectavel
sub_a4 = df_clean[df_clean['cliente_id'] == 'CLI-A-4']
limite_a4 = sub_a4['valor_brl'].median() * 5
op = sub_a4[sub_a4['id'] == 'OP-0013'].iloc[0]
print(f"\nOP-0013 em BRL convertido: {op['valor_brl']:.2f} > limite {limite_a4:.2f} -> atipico")
print(f"OP-0013 sem converter:     {op['valor']:.2f} > limite {limite_a4:.2f} -> "
      f"{'atipico' if op['valor'] > limite_a4 else 'NAO seria detectado'}")

,qtd_operacoes,mediana_brl,maior_valor_brl,limite_5x,elegivel_4_ops,sinalizado
cliente_id,,,,,,
CLI-A-1,4,17700.0,18800.0,88500.0,True,False
CLI-A-2,2,26450.0,27000.0,132250.0,False,False
CLI-A-3,3,16100.0,17200.0,80500.0,False,False
CLI-A-4,4,5450.0,64800.0,27250.0,True,True
CLI-A-5,4,3600.0,7000.0,18000.0,True,False
CLI-A-6,2,5100.0,8800.0,25500.0,False,False


OK: Regra 2 sinaliza CLI-A-4 e nao sinaliza CLI-A-2 (valor alto porem tipico)
OK: CLI-A-3 fica fora da regra por ter menos de 4 operacoes apos a limpeza

OP-0013 em BRL convertido: 64800.00 > limite 27250.00 -> atipico
OP-0013 sem converter:     12000.00 > limite 27250.00 -> NAO seria detectado


# Parte B — Análise com LLM

Escolhemos **`CLI-A-1`**, sinalizado pela Regra 1 (fracionamento): 3 transferências no mesmo
dia para a mesma contraparte, cada uma abaixo do limite de reporte de R$ 20.000,00 — um padrão
clássico de *structuring*. O papel do LLM aqui é **interpretar e redigir o parecer**, não
recalcular nada — soma, contagem e comparação com limite já foram feitas em pandas acima.

In [14]:
import re

class ParecerLLM(BaseModel):
    nivel_risco: Literal['baixo', 'medio', 'alto']
    tipologia_suspeita: str
    red_flags: list[str]
    justificativa: str


def montar_contexto_cliente(df_clean, cliente_id):
    sub = df_clean[df_clean['cliente_id'] == cliente_id].copy()
    sub['data'] = sub['data'].dt.strftime('%Y-%m-%d')
    cols = ['id', 'data', 'valor_brl', 'canal', 'tipo', 'contraparte']
    return sub[cols].to_dict(orient='records')


cliente_alvo = 'CLI-A-1'
contexto = montar_contexto_cliente(df_clean, cliente_alvo)
flags_deterministicas = {
    'flag_fracionamento': bool(df_clean.loc[df_clean['cliente_id'] == cliente_alvo, 'flag_fracionamento'].any()),
    'flag_valor_atipico': bool(df_clean.loc[df_clean['cliente_id'] == cliente_alvo, 'flag_valor_atipico'].any()),
}
contexto, flags_deterministicas

([{'id': 'OP-0001',
   'data': '2026-03-09',
   'valor_brl': 18100.0,
   'canal': 'pix',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Alfa Comercio LTDA'},
  {'id': 'OP-0002',
   'data': '2026-03-09',
   'valor_brl': 17300.0,
   'canal': 'pix',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Alfa Comercio LTDA'},
  {'id': 'OP-0003',
   'data': '2026-03-09',
   'valor_brl': 18800.0,
   'canal': 'ted',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Beta Servicos ME'},
  {'id': 'OP-0004',
   'data': '2026-03-21',
   'valor_brl': 3300.0,
   'canal': 'boleto',
   'tipo': 'pagamento',
   'contraparte': 'Gama Distribuidora'}],
 {'flag_fracionamento': True, 'flag_valor_atipico': False})

In [15]:
from dotenv import load_dotenv
from groq import Groq

load_dotenv('../.env')
client = Groq(api_key=os.environ['GROQ_API_KEY'])
MODEL = os.environ.get('LLM_MODEL', 'openai/gpt-oss-120b')

SCHEMA_HINT = '''Responda APENAS com um JSON válido, sem texto antes ou depois, no formato:
{
  "nivel_risco": "baixo" | "medio" | "alto",
  "tipologia_suspeita": "string curta",
  "red_flags": ["lista de strings"],
  "justificativa": "string"
}'''


def chamar_llm(prompt_sistema, prompt_usuario):
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': prompt_sistema},
            {'role': 'user', 'content': prompt_usuario},
        ],
        max_tokens=1200,
        temperature=0.2,
    )
    latencia = time.time() - t0
    texto = resp.choices[0].message.content or ''
    usage = resp.usage
    return {
        'texto': texto,
        'tokens_prompt': usage.prompt_tokens,
        'tokens_resposta': usage.completion_tokens,
        'tokens_total': usage.total_tokens,
        'latencia_s': round(latencia, 2),
    }


def parsear_parecer(texto):
    """Extrai e valida o JSON da resposta; trata resposta malformada."""
    match = re.search(r'\{.*\}', texto, re.DOTALL)
    if not match:
        return None, 'nenhum JSON encontrado na resposta'
    try:
        obj = json.loads(match.group(0))
    except json.JSONDecodeError as e:
        return None, f'JSON invalido: {e}'
    try:
        parecer = ParecerLLM(**obj)
    except ValidationError as e:
        return None, f'schema invalido: {e}'
    return parecer, None

### Prompt v1 — direto, pouco contexto de negócio

In [16]:
prompt_sistema_v1 = 'Voce e um analista de compliance bancario.'
prompt_usuario_v1 = f'''Cliente {cliente_alvo} foi sinalizado por fracionamento.
Operacoes: {json.dumps(contexto, ensure_ascii=False)}
{SCHEMA_HINT}'''

resultado_v1 = chamar_llm(prompt_sistema_v1, prompt_usuario_v1)
parecer_v1, erro_v1 = parsear_parecer(resultado_v1['texto'])

print('--- métricas ---')
print({k: v for k, v in resultado_v1.items() if k != 'texto'})
print('--- erro de parsing ---', erro_v1)
print('--- parecer ---')
parecer_v1

--- métricas ---
{'tokens_prompt': 410, 'tokens_resposta': 312, 'tokens_total': 722, 'latencia_s': 1.07}
--- erro de parsing --- None
--- parecer ---


ParecerLLM(nivel_risco='alto', tipologia_suspeita='Fracionamento de valores (structuring)', red_flags=['Três transferências enviadas no mesmo dia com valores próximos ao limite de 20 mil reais', 'Uso repetido do canal PIX para valores elevados', 'Transações para a mesma contraparte (Alfa Comercio LTDA) em sequência', 'Distribuição de valores em múltiplas operações para evitar reporte'], justificativa='O cliente realizou várias transferências de valores próximos ao teto de reporte (R$ 20.000) no mesmo dia, utilizando principalmente o canal PIX, o que indica tentativa de dividir o montante para fugir da comunicação obrigatória. Essa prática caracteriza fracionamento, elevando o risco para alto e demandando investigação adicional.')

### Prompt v2 — contexto de negócio explícito, papel definido, regras já calculadas
informadas (para o LLM não tentar recalcular) e pedido de rigor na justificativa

In [17]:
prompt_sistema_v2 = '''Voce e um analista senior de Prevencao a Lavagem de Dinheiro (PLD/AML)
de um banco. Voce recebe operacoes ja filtradas e sinalizadas por regras deterministicas
(calculo ja foi feito fora deste prompt). Sua funcao e SOMENTE interpretar o padrao e redigir
um parecer tecnico, nunca recalcular somas, medias ou comparar com limites - isso ja foi feito.'''

prompt_usuario_v2 = f'''Cliente: {cliente_alvo}

Flags deterministicas ja calculadas (nao recalcule, apenas interprete):
{json.dumps(flags_deterministicas, ensure_ascii=False)}

Regra que disparou (fracionamento): 3+ operacoes na mesma data cuja soma ultrapassa
R$ 50.000,00, sem nenhuma operacao isolada atingir R$ 20.000,00 - padrao classico de
structuring para evitar limites de reporte.

Operacoes do cliente:
{json.dumps(contexto, ensure_ascii=False)}

Escreva um parecer tecnico contextualizado (considere canal, contraparte repetida, tipo de
operacao). {SCHEMA_HINT}'''

resultado_v2 = chamar_llm(prompt_sistema_v2, prompt_usuario_v2)
parecer_v2, erro_v2 = parsear_parecer(resultado_v2['texto'])

print('--- métricas ---')
print({k: v for k, v in resultado_v2.items() if k != 'texto'})
print('--- erro de parsing ---', erro_v2)
print('--- parecer ---')
parecer_v2

--- métricas ---
{'tokens_prompt': 604, 'tokens_resposta': 437, 'tokens_total': 1041, 'latencia_s': 1.17}
--- erro de parsing --- None
--- parecer ---


ParecerLLM(nivel_risco='alto', tipologia_suspeita='Structuring (smurfing) de pagamentos', red_flags=['Fracionamento de valores em múltiplas operações no mesmo dia', 'Soma das operações ultrapassa R$ 50.000,00 sem nenhuma atingir R$ 20.000,00 individualmente', 'Uso de canais eletrônicos (PIX, TED) para valores próximos ao limite de reporte', 'Contraparte recorrente (Alfa Comercio LTDA) em duas das três operações fracionadas'], justificativa='No dia 09/03/2026 o cliente realizou três transferências enviadas (duas via PIX para Alfa Comercio LTDA e uma via TED para Beta Servicos ME) totalizando R$ 54.200,00, todas abaixo de R$ 20.000,00. Esse padrão corresponde ao critério de fracionamento definido (≥3 operações na mesma data, soma > R$ 50.000,00, sem operação isolada acima de R$ 20.000,00), típico de tentativa de evitar o reporte obrigatório. A presença de múltiplos canais eletrônicos e a repetição de contraparte reforçam a suspeita de structuring para ocultar a origem ou destino dos recu

### Comparação entre as duas versões (com base na execução acima)

Nesta execução **as duas versões chegaram a `nivel_risco: alto`**, que é a classificação
correta. A diferença está na *qualidade e na estabilidade* do raciocínio, não no rótulo final.

**v2 é mais preciso e mais verificável.** Ele cita os números exatos — três transferências em
09/03/2026 somando R$ 54.200,00, todas abaixo de R$ 20.000,00 — e nomeia a tipologia como
*structuring/smurfing*. Ele acerta porque esses números foram **entregues** a ele junto com a
regra que disparou; não precisou inferir nada. v1 chega ao mesmo nível de risco, mas descreve o
padrão de forma mais vaga e sem ancorar em valores conferíveis.

**O achado mais importante não está nesta execução isolada, e sim na comparação entre
execuções.** O notebook foi executado duas vezes durante o desenvolvimento, com o mesmo prompt
v1 e `temperature=0.2`. Na primeira execução (preservada no histórico do git, commit `9a59dd8`)
o v1 **alucinou um limite de reporte de "≈ R$ 10.000"** — número que não existe no enunciado nem
nos dados, onde o limite real é R$ 20.000,00. Nesta segunda execução, o mesmo prompt citou
R$ 20.000 corretamente.

Ou seja: **o prompt v1 não é errado, é instável.** Por ser subespecificado, ele deixa o modelo
preencher a lacuna do limite de reporte com conhecimento genérico sobre PLD — e às vezes esse
preenchimento é um número plausível porém falso. O v2 não tem essa lacuna para preencher, porque
o valor vem no prompt como fato.

**A conclusão prática**: o ganho do v2 não vem de ser "melhor escrito". Vem de **remover do
modelo a oportunidade de inventar um número**. É a mesma separação regra/LLM do resto do
projeto, aplicada dentro do prompt — o cálculo (R$ 20.000,00, R$ 54.200,00, 3 operações) vem do
pandas, e o modelo só interpreta e redige.

**Custo dessa precisão**: v2 consome mais tokens (1.041 contra 722, ~44% a mais), porque o
contexto extra vai no prompt. A latência é praticamente igual. Para um caso, é irrelevante; num
lote grande é o trade-off explícito entre precisão e custo — e, dado que o erro do v1 é inventar
um limite regulatório, pagar 44% a mais em tokens é barato.

In [18]:
comparacao = pd.DataFrame([
    {'versao': 'v1', **{k: v for k, v in resultado_v1.items() if k != 'texto'},
     'nivel_risco': parecer_v1.nivel_risco if parecer_v1 else None},
    {'versao': 'v2', **{k: v for k, v in resultado_v2.items() if k != 'texto'},
     'nivel_risco': parecer_v2.nivel_risco if parecer_v2 else None},
])
comparacao

,versao,tokens_prompt,tokens_resposta,tokens_total,latencia_s,nivel_risco
0,v1,410,312,722,1.07,alto
1,v2,604,437,1041,1.17,alto
